# ML Zoomcamp 2026 — Homework 1: Intro / Environment & Pandas Basics

**Course:** [ML Zoomcamp 2026](https://courses.datatalks.club/ml-zoomcamp-2026/)
**Module:** 01-intro
**Homework spec:** [homework.md](https://github.com/DataTalksClub/machine-learning-zoomcamp/blob/main/cohorts/2026/homework/01-intro/homework.md).  
**Author:** Somasekhar Reddy     **Date:** 14/09/2026. 

## 0. Goal
Get comfortable with the environment (NumPy, Pandas, Matplotlib, Seaborn) and practice
basic exploratory data analysis and a from-scratch linear regression via the normal
equation, using the 2026 Car Fuel Efficiency dataset.

**Dataset:** [car_fuel_efficiency_2026.csv](https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv)

## Questions covered in this notebook
1. Pandas version
2. Records count
3. Fuel types
4. Missing values
5. Max fuel efficiency (Asia)
6. Median horsepower before/after imputation
7. Sum of linear regression weights (normal equation)

---

## 1.0 Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 1.1 Import Data

In [ ]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
df = pd.read_csv(url)

In [3]:
df.head()


,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [4]:
df.describe(include='all')

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
count,10000.000000,10000,10000,10000,10000.00000,10000.000000,10000.000000,9123.000000,10000.000000,9736.000000,10000.000000
unique,NaN,3,3,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,USA,Gasoline,Front-wheel drive,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,4202,6913,5519,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,1999.549500,NaN,NaN,NaN,3.50400,2268.807000,6.035200,254.446016,4286.754000,18.888044,29.997700
std,14.207925,NaN,NaN,NaN,0.95209,183.447355,0.542578,22.844613,266.212491,1.279543,2.930245
min,1975.000000,NaN,NaN,NaN,2.00000,1590.000000,4.000000,171.000000,3320.000000,14.000000,19.800000
25%,1987.000000,NaN,NaN,NaN,3.00000,2150.000000,6.000000,239.000000,4110.000000,18.000000,28.000000
50%,2000.000000,NaN,NaN,NaN,4.00000,2270.000000,6.000000,254.000000,4280.000000,18.900000,30.000000
75%,2012.000000,NaN,NaN,NaN,4.00000,2390.000000,6.000000,270.000000,4470.000000,19.700000,31.900000


---

## 2. EDA

### Q1. Pandas version
What version of Pandas did you install?

In [5]:
pd.__version__

'3.0.5'

### Q2. Records count
How many records are in the dataset?

In [6]:
df.shape

(10000, 11)

##### The dataset has 10k rows and 11 features

### Q3. Fuel types
How many fuel types are presented in the dataset?

In [7]:
df["fuel_type"].value_counts()

fuel_type
Gasoline    6913
Diesel      2053
Hybrid      1034
Name: count, dtype: int64

##### There are 3 fuel types : Gasoline, Diesel and Hybrid

### Q4. Missing values
How many columns in the dataset have missing values?

In [8]:
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

##### There are null values in 2 columns: horsepower and acceleration

### Q5. Max fuel efficiency
What's the maximum fuel efficiency of cars from Asia?

In [9]:
df.groupby(["origin"]).agg({"fuel_efficiency_mpg": ["max", "min", "mean"]})    

fuel_efficiency_mpg                 
                       max   min       mean
origin                                     
Asia                  41.2  21.3  30.741015
Europe                39.4  20.7  30.552675
USA                   38.2  19.8  29.097739

##### Maximum fuel efficiency of cars from Asia is 41.2

### Q6. Median value of horsepower
After filling missing horsepower values with the most frequent value, has the median changed?
1. Find the median value of the `horsepower` column in the dataset.
2. Next, calculate the most frequent value of the same `horsepower` column.
3. Use the `fillna` method to fill the missing values in the `horsepower` column with the most frequent value from the previous step.
4. Now, calculate the median value of `horsepower` once again.

In [10]:
hp_median = df["horsepower"].median() # mdeian
hp_mode = df["horsepower"].mode()[0] # mode

df["horsepower_filled"] = df["horsepower"].fillna(hp_mode) # fill null values with mode

hp_median_new = df["horsepower_filled"].median() # median after filling null values

# Print the results
print(f"Median of horsepower: {hp_median}")
print(f"Mode of horsepower: {hp_mode}")
print(f"Median of horsepower (filled): {hp_median_new}")

Median of horsepower: 254.0
Mode of horsepower: 252.0
Median of horsepower (filled): 252.0


##### Median before null value imputation is 254 and after imputation strategy has changed to 252.

### Q7. Sum of weights
Linear regression via matrix operations — what's the sum of the result elements?

1. Select all the cars from Asia
2. Select only columns `vehicle_weight` and `model_year`
3. Select the first 7 values
4. Get the underlying NumPy array. Let's call it `X`.
5. Compute matrix-matrix multiplication between the transpose of `X` and `X`. To get the transpose, use `X.T`. Let's call the result `XTX`.
6. Invert `XTX`.
7. Create an array `y` with values `[1100, 1300, 800, 900, 1000, 1100, 1200]`.
8. Multiply the inverse of `XTX` with the transpose of `X`, and then multiply the result by `y`. Call the result `w`.
9. What's the sum of all the elements of the result?


**🔑 Key formula:**
$$
\boldsymbol{w = (X^T X)^{-1} X^T y}
$$




In [11]:
# Select the first 7 rows where origin is "Asia" and subset columns "vehicle_weight" and "model_year"
df_asian_weight_year_7 = df[df["origin"]=="Asia"][["vehicle_weight","model_year"]].iloc[:7]

# Convert the DataFrame to a NumPy array
X = np.array(df_asian_weight_year_7)

# Transpose the array
XT = X.T 

# Compute the dot product of XT and X
XTX = XT.dot(X)

# Compute the inverse of XTX
XTX_inv = np.linalg.inv(XTX)    

# Convert the list into array
listy = [1100, 1300, 800, 900, 1000, 1100, 1200]
y=np.array(listy)

# Inspect the shapes of the matrices - matrix multiplication is only possible if the inner dimensions match. 
# For example, if A is an m x n matrix and B is an n x p matrix, then the product AB is defined and will be an m x p matrix. 
# If the inner dimensions do not match (i.e., n in this case), then the multiplication is not defined.
print(f"Shape of XTXI: {XTX_inv.shape}")
print(f"Shape of XTX: {XTX.shape}")
print(f"Shape of XT: {XT.shape}")
print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")

# Compute the weights using the normal equation: w = (X^T * X)^-1 * X^T * y
w = XTX_inv.dot(XT).dot(y)

# Compute the sum of the weights
w.sum()


Shape of XTXI: (2, 2)
Shape of XTX: (2, 2)
Shape of XT: (2, 7)
Shape of X: (7, 2)
Shape of y: (7,)


np.float64(0.36919696904927746)

##### Sum of weights is roughly 0.369